In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

# Mengatur agar plot ditampilkan dengan baik
%matplotlib inline
plt.style.use('fivethirtyeight')

2025-11-16 21:38:03.709134: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-16 21:38:03.709406: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-16 21:38:03.740050: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-16 21:38:04.360821: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

In [2]:
# Tentukan Ticker
target_ticker = '^JKSE'
exogenous_tickers = ['^GSPC', '^N225', 'IDR=X', 'CL=F'] # S&P 500, Nikkei, USD/IDR, Crude Oil

# Tentukan rentang waktu (misal: 5 tahun terakhir)
start_date = '2019-01-01'
end_date = pd.to_datetime('today').strftime('%Y-%m-%d')

# Unduh data
all_tickers = [target_ticker] + exogenous_tickers
data = yf.download(all_tickers, start=start_date, end=end_date)['Adj Close']

# Ganti nama kolom agar lebih mudah dibaca
data = data.rename(columns={
    target_ticker: 'IHSG',
    '^GSPC': 'SP500',
    '^N225': 'Nikkei',
    'IDR=X': 'USDIDR',
    'CL=F': 'Oil'
})

# Menangani data yang hilang (missing values) karena hari libur pasar
# Forward fill adalah metode yang umum untuk data time series
data = data.fillna(method='ffill').dropna()

print("Data Head:")
print(data.head())

print("\nData Tail:")
print(data.tail())

/tmp/ipykernel_105428/1682828781.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(all_tickers, start=start_date, end=end_date)['Adj Close']
[*********************100%***********************]  5 of 5 completed


KeyError: 'Adj Close'

In [ ]:
# Tentukan variabel target (y) dan eksternal (X)
y = data['IHSG']
X = data.drop('IHSG', axis=1)

# Bagi data, 90% untuk training, 10% untuk testing
split_percentage = 0.9
split_index = int(len(data) * split_percentage)

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

print(f"Jumlah data training: {len(y_train)}")
print(f"Jumlah data testing: {len(y_test)}")

In [ ]:
# Temukan model ARIMA terbaik secara otomatis
# m=1 menunjukkan tidak ada seasonalitas harian (default)
# stepwise=True mempercepat pencarian
arimax_model = auto_arima(
    y_train,
    exogenous=X_train,
    stepwise=True,
    suppress_warnings=True,
    trace=True,
    seasonal=False,
    error_action='ignore'
)

print("\n--- Ringkasan Model ARIMAX ---")
print(arimax_model.summary())

# Dapatkan residual (error) dari model training
# Ini adalah bagian non-linear yang akan dipelajari LSTM
arimax_residuals_train = pd.Series(arimax_model.resid(), index=y_train.index)

In [ ]:
# 1. Scaling Residual
scaler = MinMaxScaler(feature_range=(0, 1))
# Ubah bentuk data agar sesuai dengan scaler
residuals_train_reshaped = arimax_residuals_train.values.reshape(-1, 1)
scaled_residuals_train = scaler.fit_transform(residuals_train_reshaped)

# 2. Membuat Sekuens Data
# look_back adalah jumlah hari sebelumnya yang digunakan untuk memprediksi hari berikutnya
look_back = 60

# Gunakan TimeseriesGenerator untuk membuat data sekuens dengan mudah
# Ini akan membuat batch (X, y)
# di mana X adalah 60 residual sebelumnya, dan y adalah residual ke-61
generator = TimeseriesGenerator(
    scaled_residuals_train,
    scaled_residuals_train,
    length=look_back,
    batch_size=1
)

# Konversi generator ke list untuk X dan y
X_lstm_train = []
y_lstm_train = []
for i in range(len(generator)):
    x_batch, y_batch = generator[i]
    X_lstm_train.append(x_batch)
    y_lstm_train.append(y_batch)

# Ubah ke format numpy array yang benar untuk LSTM [samples, timesteps, features]
X_lstm_train = np.array(X_lstm_train).reshape(-1, look_back, 1)
y_lstm_train = np.array(y_lstm_train).reshape(-1, 1)

print(f"Bentuk X_lstm_train: {X_lstm_train.shape}")
print(f"Bentuk y_lstm_train: {y_lstm_train.shape}")

In [ ]:
# Membangun Model LSTM
model_lstm = Sequential()
model_lstm.add(LSTM(
    units=50,
    return_sequences=True,
    input_shape=(look_back, 1)
))
model_lstm.add(Dropout(0.2))
model_lstm.add(LSTM(units=50))
model_lstm.add(Dropout(0.2))
model_lstm.add(Dense(units=1)) # Output layer

# Compile model
model_lstm.compile(optimizer='adam', loss='mean_squared_error')

# Tampilkan ringkasan model
model_lstm.summary()

# Latih model LSTM
# (Epochs bisa ditambah untuk akurasi lebih baik, tapi 25-50 sudah cukup untuk awal)
print("\n--- Melatih Model LSTM ---")
history = model_lstm.fit(
    X_lstm_train,
    y_lstm_train,
    epochs=25,
    batch_size=32,
    verbose=1,
    shuffle=False # Penting untuk data time series
)

In [ ]:
print("\n--- Memulai Prediksi Walk-Forward ---")

# Dapatkan seluruh residual (training) yang akan digunakan untuk input awal LSTM
current_residuals = list(scaled_residuals_train[-look_back:])
hybrid_predictions = []

for i in range(len(y_test)):
    # 1. Prediksi ARIMAX
    # Prediksi 1 langkah ke depan menggunakan data eksternal (X_test)
    arimax_pred = arimax_model.predict(
        n_periods=1,
        exogenous=X_test.iloc[i:i+1]
    )[0] # Ambil nilai prediksinya

    # 2. Prediksi LSTM (Residual)
    # Siapkan input untuk LSTM (60 residual terakhir)
    lstm_input = np.array(current_residuals).reshape(1, look_back, 1)

    # Lakukan prediksi
    lstm_pred_scaled = model_lstm.predict(lstm_input, verbose=0)

    # Kembalikan ke skala semula
    lstm_pred = scaler.inverse_transform(lstm_pred_scaled)[0, 0]

    # 3. Prediksi Hybrid (Gabungan)
    final_pred = arimax_pred + lstm_pred
    hybrid_predictions.append(final_pred)

    # 4. Update Model
    # Dapatkan nilai aktual (true value)
    true_value = y_test.iloc[i]
    true_exog = X_test.iloc[i:i+1]

    # Hitung residual baru berdasarkan prediksi ARIMAX
    # (Bukan prediksi hybrid, karena LSTM hanya memodelkan error ARIMAX)
    new_residual = true_value - arimax_pred

    # Ubah skala residual baru
    new_residual_scaled = scaler.transform(np.array([[new_residual]]))[0, 0]

    # Perbarui model ARIMAX dengan data aktual
    arimax_model.update(true_value, exogenous=true_exog)

    # Perbarui daftar residual untuk input LSTM berikutnya
    current_residuals.pop(0) # Hapus yang tertua
    current_residuals.append(new_residual_scaled) # Tambah yang terbaru

    if (i+1) % 50 == 0 or (i+1) == len(y_test):
        print(f"Memprediksi langkah {i+1}/{len(y_test)}...")

print("Prediksi Selesai.")

# Ubah hasil prediksi menjadi Pandas Series untuk plotting
predictions_series = pd.Series(hybrid_predictions, index=y_test.index)

In [ ]:
# Hitung RMSE
rmse = np.sqrt(mean_squared_error(y_test, predictions_series))
print(f'\n--- Evaluasi Model ---')
print(f'Test RMSE: {rmse:.3f}')

# Plot hasil
plt.figure(figsize=(15, 7))
plt.plot(y_train, label='Data Training')
plt.plot(y_test, label='Data Aktual (Test)', color='orange')
plt.plot(predictions_series, label='Prediksi Hybrid (ARIMAX+LSTM)', color='green', linestyle='--')
plt.title(f'Prediksi IHSG (RMSE: {rmse:.3f})')
plt.xlabel('Tanggal')
plt.ylabel('Harga Penutupan IHSG')
plt.legend()
plt.show()